# Leaf area index with OpenEO

This notebook demonstrates the calculation of Leaf Area Index (LAI) using Sentinel-2 imagery with the OpenEO API.

## Overview

In this notebook, we will:
1. Connect to an OpenEO backend service
2. Define an area of interest containing vegetated areas, such as plantation, forest, cropland
3. Load Sentinel-2 L2A imagery for a specific date
4. Calculate and visualize the leaf area index algorithm
5. Export results for further analysis

## What is Leaf Area Index (LAI)?

LAI is a dimensionless index that measures the one-sided green leaf area per unit of land area (m2 / m2). Following the methodology in [Evalscript's LAI](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/lai/), this notebook implements a neural network trained on Sentinel-2 data. It:
1. Takes 8 spectral bands (B03, B04, B05, B06, B07, B8A, B11, B12) and 4 angular inputs (view/sun zenith and azimuth)
2. Normalizes all inputs to [-1, 1] range, except for sun and view azimuth angles. Sun and view azimuth angles are used to calculate relative azimuth: `cos(sunAzimuth - viewAzimuth)`
3. Passes them through 5 neurons using `tanh` activation (hyperbolic tangent sigmoid)
4. Combines neuron outputs in a final linear `layer2` 
5. Denormalizes the result back to physical units - valid range: 0.000319 to 14.47 m²/m²
6. Divides LAI by 3 (`lai / 3`), mapping the typical LAI range of 0-3 to a normalized display range of 0-1

### Applications

- Crop growth monitoring and yield estimation as LAI tracks canopy development across growing season
- Forest biomass and carbon stock estimation as LAI is a key input to carbon flux models
- Deforestation and degradation monitoring as sudden LAI drops signal disturbance
- Ecosystem productivity modelling (GPP/NPP) as LAI drives light interception calculation

## Import Required Libraries

We begin by importing the necessary Python libraries for data processing and visualization.

In [ ]:
import ipynbname
import shapely
import rioxarray
import matplotlib.pyplot as plt
from openeo.processes import pi, cos, divide, tanh
from openeo.api.process import Parameter

# OpenEO UDP parameter management system
from openeo_udp import ParameterManager

## Load Parameters and Connect to OpenEO Backend

Load algorithm parameters from the co-located parameter file and connect to an OpenEO backend with automatic endpoint selection.

In [ ]:
# Initialize the algorithm ID for UDP registration
_algorithm_id = ipynbname.name()

# Initialize parameter manager
param_manager = ParameterManager('leaf_area_index.params.py')

# Display available options using the built-in helper
param_manager.print_options("Leaf area index algorithm")

# Note: Connection will be established using the interactive widgets below
# This avoids authentication issues and ensures proper endpoint-specific handling

In [ ]:
# Connect using the a parameter set for a specified location on the Copernicus Data Space endpoint
connection, current_params = param_manager.quick_connect(
    param_set="cropland_plain_croatia",
    endpoint="copernicus_dataspace",
)

## Load Sentinel-2 Data

Load Sentinel-2 L2A (atmospherically corrected) data. We need specific bands for leaf area index calculation:

- **B03** (559.8 nm): Green band
- **B04** (664.6 nm): Red band
- **B05** (704.1 nm): Vegetation red edge
- **B06** (740.5 nm): Vegetation red edge
- **B07** (782.8 nm): Vegetation red edge
- **B8A** (832.8 nm): NIR
- **B11** (1613.7 nm): SWIR
- **B12** (2202.4 nm): SWIR

In [ ]:
# Load Sentinel-2 data using the selected parameters
s2cube = connection.load_collection(
    current_params["collection"].default,
    temporal_extent=current_params["time"].default,
    spatial_extent=current_params["bounding_box"].default,
    bands=current_params["bands"].default,
    properties={
        "eo:cloud_cover": lambda x: x <= current_params["cloud_cover"].default,
    },
)

# Avoid extra data outside the spatial extent
s2cube = s2cube.mask_polygon(
    shapely.geometry.box(
        minx=current_params["bounding_box"].default["west"],
        miny=current_params["bounding_box"].default["south"],
        maxx=current_params["bounding_box"].default["east"],
        maxy=current_params["bounding_box"].default["north"]
    )
)

# Apply temporal reduction (modify as needed for your algorithm)
s2cube = s2cube.reduce_dimension(dimension="t", reducer="first")

print("✅ Sentinel-2 data loaded successfully!")

## Implement Leaf Area Index Algorithm

This function implements the leaf area index algorithm:

1. Takes 8 spectral bands (B03, B04, B05, B06, B07, B8A, B11, B12) and 4 angular inputs (view/sun zenith and azimuth)
2. Normalizes all inputs to [-1, 1] range, except for sun and view azimuth angles. Sun and view azimuth angles are used to calculate relative azimuth: `cos(sunAzimuth - viewAzimuth)`
3. Passes them through 5 neurons from SNAP using `tanh` activation (hyperbolic tangent sigmoid)
4. Combines neuron outputs in a final linear `layer2` 
5. Denormalizes the result back to physical units - valid range: 0.000319 to 14.47 m²/m²
6. Divides LAI by 3 (`lai / 3`), mapping the typical LAI range of 0-3 to a normalized display range of 0-1

In [ ]:
def normalize(unnormalized, min, max):
    return 2 * (unnormalized - min) / (max - min) - 1

In [ ]:
def denormalize(normalized, min, max):
    return 0.5 * (normalized + 1) * (max - min) + min

In [ ]:
def neuron1(input_list):	
	weights = [
		-0.023406878966470,
		+0.921655164636366,
		+0.135576544080099,
		-1.938331472397950,
		-3.342495816122680,
		+0.902277648009576,
		+0.205363538258614,
		-0.040607844721716,
		-0.083196409727092,
		+0.260029270773809,
		+0.284761567218845,
	]
	s = 4.96238030555279
	for w, x in zip(weights, input_list):
		s = s + w * x
	
	return tanh(s)

In [ ]:
def neuron2(input_list):
    weights = [
        -0.132555480856684,
        -0.139574837333540,
        -1.014606016898920,
        -1.330890038649270,
        +0.031730624503341,
        -1.433583541317050,
        -0.959637898574699,
        +1.133115706551000,
        +0.216603876541632,
        +0.410652303762839,
        +0.064760155543506,
    ]

    s = 1.416008443981500
    for w, x in zip(weights, input_list):
        s = s + w * x

    return tanh(s)

In [ ]:
def neuron3(input_list):
    weights = [
        +0.086015977724868,
        +0.616648776881434,
        +0.678003876446556,
        +0.141102398644968,
        -0.096682206883546,
        -1.128832638862200,
        +0.302189102741375,
        +0.434494937299725,
        -0.021903699490589,
        -0.228492476802263,
        -0.039460537589826,
    ]
    
    s = 1.075897047213310
    for w, x in zip(weights, input_list):
        s = s + w * x

    return tanh(s)

In [ ]:
def neuron4(input_list):
    weights = [
        -0.109366593670404,
        -0.071046262972729,
        +0.064582411478320,
        +2.906325236823160,
        -0.673873108979163,
        -3.838051868280840,
        +1.695979344531530,
        +0.046950296081713,
        -0.049709652688365,
        +0.021829545430994,
        +0.057483827104091
    ]

    s = 1.533988264655420
    for w, x in zip(weights, input_list):
        s = s + w * x

    return tanh(s)

In [ ]:
def neuron5(input_list):
    weights = [
        -0.089939416159969,
        +0.175395483106147,
        -0.081847329172620,
        +2.219895367487790,
        +1.713873975136850,
        +0.713069186099534,
        +0.138970813499201,
        -0.060771761518025,
        +0.124263341255473,
        +0.210086140404351,
        -0.183878138700341
    ]

    s = 3.024115930757230
    for w, x in zip(weights, input_list):
        s = s + w * x

    return tanh(s) 

In [ ]:
def layer2(neuron_arr_1, neuron_arr_2, neuron_arr_3, neuron_arr_4, neuron_arr_5):
    weights = [
        -1.500135489728730,
        -0.096283269121503,
        -0.194935930577094,
        -0.352305895755591,
        +0.075107415847473
    ]
    neuron_list = [
        neuron_arr_1,
        neuron_arr_2,
        neuron_arr_3,
        neuron_arr_4,
        neuron_arr_5
    ]

    s = 1.096963107077220
    for w, x in zip(weights, neuron_list):
        s = s + w * x

    return s

In [ ]:
# Scale factor to convert backend-native band values to 0-1 reflectance.
# Provided by the endpoint mapper: 1.0 for endpoints that already return reflectance
# (e.g. eopf_explorer), 10000.0 for integer-scaled L2A (e.g. copernicus_dataspace).
reflectance_scale = current_params["reflectance_scale"]


def lai_main_algorithm(data):
    B03, B04, B05, B06, B07, B8A, B11, B12, viewZenithMean, viewAzimuthMean, sunZenithAngles, sunAzimuthAngles = (
        data[0],
        data[1],
        data[2],
        data[3],
        data[4],
        data[5],
        data[6],
        data[7],
        data[8],
        data[9],
        data[10],
        data[11]
    )

    B03 = B03 / reflectance_scale
    B04 = B04 / reflectance_scale
    B05 = B05 / reflectance_scale
    B06 = B06 / reflectance_scale
    B07 = B07 / reflectance_scale
    B8A = B8A / reflectance_scale
    B11 = B11 / reflectance_scale
    B12 = B12 / reflectance_scale

    degToRad = divide(pi(), 180)

    B03_norm = normalize(B03, 0, 0.253061520471542)
    B04_norm = normalize(B04, 0, 0.290393577911328)
    B05_norm = normalize(B05, 0, 0.305398915248555)
    B06_norm = normalize(B06, 0.006637972542253, 0.608900395797889)
    B07_norm = normalize(B07, 0.013972727018939, 0.753827384322927)
    B8A_norm = normalize(B8A, 0.026690138082061, 0.782011770669178)
    B11_norm = normalize(B11, 0.016388074192258, 0.493761397883092)
    B12_norm = normalize(B12, 0, 0.493025984460231)
    viewZenith_norm = normalize(cos(viewZenithMean * degToRad), 0.918595400582046, 1)
    sunZenith_norm = normalize(cos(sunZenithAngles * degToRad), 0.342022871159208, 0.936206429175402)
    relAzimuth_norm = cos((sunAzimuthAngles - viewAzimuthMean) * degToRad)

    input_band = [
        B03_norm,
        B04_norm,
        B05_norm,
        B06_norm,
        B07_norm,
        B8A_norm,
        B11_norm,
        B12_norm,
        viewZenith_norm,
        sunZenith_norm,
        relAzimuth_norm,
    ]

    n1 = neuron1(input_band)
    n2 = neuron2(input_band)
    n3 = neuron3(input_band)
    n4 = neuron4(input_band)
    n5 = neuron5(input_band)

    l2 = layer2(n1, n2, n3, n4, n5)

    lai = denormalize(l2, 0.000319182538301, 14.4675094548151)

    return lai / 3 

## Apply Algorithm

Apply the Leaf area index function to our Sentinel-2 data cube.

In [ ]:
# Apply algorithm on the bands dimension
lai_image = s2cube.apply_dimension(dimension="bands", process=lai_main_algorithm)

# Save as appropriate format for visualization
lai_image = lai_image.save_result("GTiff")

## Download and Visualize Results

Download a sample area and display the Leaf area index results.

In [ ]:
# Define parameters for the process graph
filename = f"{_algorithm_id}_{current_params['location_name'].replace(' ', '_').replace(',', '').lower()}.tif"

# Synchronous execution (POST /result) does not accept unresolved Parameter refs,
# so materialize them with the current parameter set's defaults before download.
# The underlying parameterized graph is preserved for the UDP export cell below.
resolved = param_manager.resolve(lai_image, current_params)
resolved.download(filename)

In [ ]:
ds = rioxarray.open_rasterio(filename).squeeze()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(
    ds,
    extent=[ds.x.values.min(), ds.x.values.max(), ds.y.values.min(), ds.y.values.max()],
    cmap="gray",
    vmin=0,
    vmax=1,
)
plt.colorbar(im, ax=ax, label="LAI/3", fraction=0.03, pad=0.02)
ax.set_title(
    f"Leaf Area Index\n{current_params['location_name']}",
    fontsize=12,
)
ax.axis("off")
plt.tight_layout()
plt.show()

## Interpretation Guide

### Leaf Area Index Results:

**Darker pixels (low values → low LAI)**
- Bare soil, sand, urban surfaces, water
- Sparse or stressed vegetation with little leaf coverage
- Harvested or unused fields

**Lighter pixels (high values → high LAI)**
- Dense, healthy vegetation such as forests, mature crops, plantations
- High canopy cover
- LAI close to or above 3 m²/m² (clipped to white at LAI ≥ 3)

**Mid-grey tones**
- Transitional vegetation such as early-season crops, grasslands, shrublands

Note: The visualization displays the normalized LAI (`LAI / 3`), meaning that pixel values with a range 0-3 are mapped to 0-1. A pixel value of 0.5, for example, corresponds to LAI = 1.5 m²/m². To display true LAI values on the colorbar, multiply the array by 3 before plotting or adjust the colorbar ticks accordingly.

### Limitations:

- The neural network is ported from SNAP without input/output validation. Out-of-range reflectance values (clouds, shadows, water) are not flagged.
- All dense canopies with LAI >= 3 (mature forest, double-cropped fields) are clipped to the same maximum display value
- The neural model was trained on specific vegetation types, results may be unreliable for surface types outside the training distribution
- Reflectance scale factor is backend-specific (10000 for CDSE, 1.0 for EOPF) — using the wrong endpoint without the correct scale produces physically meaningless band values
- Not all openEO backends support all processes used (e.g. `tanh`, `pi`). Portability depends on backend process coverage

In [ ]:
# Create a directory to export images, UDP, and OGC API records
from pathlib import Path

_repo_root = next(p for p in Path.cwd().parents if (p / "algorithm_registration").exists())
_alg_dir = _repo_root / "algorithm_registration" / _algorithm_id
_records_dir = _alg_dir / "records"
_udp_dir = _alg_dir / "openeo_udp"

_records_dir.mkdir(parents=True, exist_ok=True)
_udp_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Export the process graph for reuse
import json

process_graph = {
    "process_graph": lai_image.flat_graph(),
    "parameters": [
        current_params["time"].to_dict(),
        current_params["bounding_box"].to_dict(),
        current_params["cloud_cover"].to_dict()
    ],
    "id": _algorithm_id,
    "summary": "Calculation of Leaf Area Index (LAI) using Sentinel-2 imagery with the OpenEO API.",
    "description": (
        "Calculates Leaf Area Index (LAI) from Sentinel-2 L2A imagery using a neural network algorithm ported from SNAP."
        "The LAI is quantified as one-sided leaf area per unit land area (m²/m²). Suitable for monitoring"
        "vegetation density across croplands, forests, and plantations."
    )
}

# Save the process graph
with open(f"{_udp_dir}/{_algorithm_id}.json", "w") as f:
    json.dump(process_graph, f, indent=2)

print(f"Process graph exported to {_udp_dir}/{_algorithm_id}.json")
print(f"Process ID: {_algorithm_id}")

In [ ]:
# Export necessary metadata to register the process graph in APEx Algorithm Catalogue 
if "_algorithm_id" not in globals():
    _algorithm_id = "leaf_area_index"

_repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").exists())
_nb_href = f"{(Path.cwd() / 'leaf_area_index.ipynb').relative_to(_repo_root)}"
_alg_dir = _repo_root / "algorithm_registration" / _algorithm_id
_records_dir = _alg_dir / "records"

_records_dir.mkdir(parents=True, exist_ok=True)
# Notebook metadata
metadata = {
    "id" : _algorithm_id,
    "title" : "Leaf Area Index",
    "preview_title" : "Leaf Area Index Calculation",
    "description" : (
        "Calculates Leaf Area Index (LAI) from Sentinel-2 L2A imagery using a neural network algorithm ported from SNAP."
        "The LAI is quantified as one-sided leaf area per unit land area (m²/m²). Suitable for monitoring"
        "vegetation density across croplands, forests, and plantations."
    ),
    "keywords" : ["Agriculture", "Vegetation", "Leaf Area Index (LAI)", "Sentinel-2"],
    "themes" : ["VEGETATION", "REMOTE SENSING", "Sentinel-2 MSI"],
    "created" : "2026-04-30T00:00:00Z",
    "updated" : "2026-04-30T00:00:00Z",
    "license" : "CC-BY-SA-4.0",
    "openeo_backend_title" : "CDSE openEO Federation",
    "openeo_backend_url" : "https://openeofed.dataspace.copernicus.eu",
    "notebook_github_location" : _nb_href,
    "collection_id" : "SENTINEL2_L2A",
    "attribution" : {
        "original_script" : "https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/lai/",
        "authors" : ["Authors not listed"],
        "source_repository" : "https://github.com/sentinel-hub/custom-scripts",
        "citation" : None
    }
}

## References and Attribution

**Original Script:** [LAI (Leaf Area Index)](https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/lai/)

**Author:** Authors not listed in the original Evalscript

**Source Repository:** [Sentinel Hub Custom Scripts](https://github.com/sentinel-hub/custom-scripts)

### OpenEO Conversion:
- **Conversion Date**: 30 April 2026
- **OpenEO Framework**: Adapted for openEO API and process graph structure
- **Backend Tested**: CDSE

## Conclusion

This notebook successfully demonstrates the conversion of the leaf area index algorithm to an OpenEO User-Defined Process. The implementation:

✅ **Maintains Scientific Accuracy**: Preserves the original algorithm's methodology

✅ **Provides Flexible Parameter Management**: Works with multiple locations, which are Cropland Plain, Istria Coast, Croatia and Natural Reserve in Rome, Italy

✅ **Follows OpenEO Standards**: Uses parameterized process graphs for reusability

✅ **Includes Comprehensive Documentation**: Provides interpretation guides and usage examples

### Key Achievements:

1. **Algorithm Implementation**: Successfully implemented leaf area index using OpenEO processes
2. **Parameter Management**: Integrated with the OpenEO UDP parameter system
3. **Multi-backend Support**: Works in the backends that provide viewing zenith & azimuth angles and sun zenith & azimuth angles
4. **Process Graph Export**: Generated reusable UDP definition

This conversion makes the leaf area index algorithm accessible to the broader OpenEO ecosystem while maintaining its scientific rigor and practical utility.